# From Notebook to Production: Telco Churn Starter Notebook

This is the intentionally collapsed version of the project: data loading, preprocessing, feature engineering, model training, and model validation all live in one place.

It is organized and useful for exploration, which is how many real data science projects begin. It is also not ready for production yet. The talk can start here, then show how this notebook becomes the Databricks MLOps pipeline in the rest of the repo.

## Notebook Flow

1. Load the Telco Customer Churn data.
2. Inspect schema, missing values, and churn balance.
3. Clean and preprocess raw columns.
4. Build customer-level features.
5. Train a churn classifier.
6. Validate the model against a simple recall gate.

The production version should split these responsibilities across tested feature code, Databricks workflows, Unity Catalog tables, MLflow tracking and registry, validation jobs, deployment, and monitoring.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

RANDOM_STATE = 29

In [ ]:
DATA_PATH_CANDIDATES = [
    Path('mlops_dbx/feature_engineering/data/telco_customer_churn.csv'),
    Path('../mlops_dbx/feature_engineering/data/telco_customer_churn.csv'),
    Path('feature_engineering/data/telco_customer_churn.csv'),
    Path('../feature_engineering/data/telco_customer_churn.csv'),
    Path('data/telco_customer_churn.csv'),
]

DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    candidates = '\n'.join(str(path) for path in DATA_PATH_CANDIDATES)
    raise FileNotFoundError(
        'Could not find telco_customer_churn.csv. Tried:\n'
        f'{candidates}\n\nCurrent working directory: {Path.cwd()}'
    )

VALIDATION_THRESHOLDS = {
    'recall': 0.50,
    'roc_auc': 0.75,
}

print(f'Using data file: {DATA_PATH.resolve()}')

## 1. Data Loading

The raw CSV is loaded directly from the repository. This is fine for a demo notebook, but in production this input would usually be a governed table with schema and quality expectations.

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
raw_df['customerID'] = raw_df['customerID'].astype('string').str.strip()
raw_df['TotalCharges'] = pd.to_numeric(raw_df['TotalCharges'], errors='coerce')
raw_df = raw_df.drop_duplicates(subset=['customerID']).reset_index(drop=True)

print(f'Loaded {raw_df.shape[0]:,} rows and {raw_df.shape[1]:,} columns')
display(raw_df.head())

In [ ]:
profile = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing_values': raw_df.isna().sum(),
    'missing_rate': raw_df.isna().mean().round(4),
    'unique_values': raw_df.nunique(dropna=True),
})

churn_balance = (
    raw_df['Churn']
    .value_counts(dropna=False)
    .rename_axis('churn_label')
    .to_frame('rows')
)
churn_balance['share'] = (churn_balance['rows'] / churn_balance['rows'].sum()).round(4)

display(profile)
display(churn_balance)

## 2. Preprocessing and Feature Engineering

This cell intentionally contains a lot of business logic. The production code in this repo moves this kind of logic into `mlops_dbx/feature_engineering/features/compute_features.py` so it can be tested, versioned, reused, and run as a Databricks workflow task.

In [ ]:
FEATURE_REQUIRED_COLUMNS = [
    'customerID',
    'gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'tenure',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod',
    'MonthlyCharges',
    'TotalCharges',
]


def require_columns(df, required_columns):
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')


def normalize_text(series):
    return series.astype('string').fillna('').str.strip()


def yes_no_to_int(series):
    return normalize_text(series).str.lower().eq('yes').astype('int64')


def to_number(series, default=None):
    values = pd.to_numeric(series, errors='coerce')
    if default is not None:
        values = values.fillna(default)
    return values


def make_labels(df_raw):
    require_columns(df_raw, ['customerID', 'Churn'])
    labels = pd.DataFrame({
        'customer_id': normalize_text(df_raw['customerID']),
        'churn': normalize_text(df_raw['Churn']).str.lower().eq('yes').astype('int64'),
    })
    return labels.drop_duplicates(subset=['customer_id']).reset_index(drop=True)


def engineer_customer_features(df_raw):
    require_columns(df_raw, FEATURE_REQUIRED_COLUMNS)

    df = pd.DataFrame({
        'customer_id': normalize_text(df_raw['customerID']),
        'gender': normalize_text(df_raw['gender']),
        'senior_citizen': to_number(df_raw['SeniorCitizen'], default=0).astype('int64'),
        'tenure_months': to_number(df_raw['tenure'], default=0).astype('int64'),
        'monthly_charges': to_number(df_raw['MonthlyCharges'], default=0.0).astype('float64'),
        'total_charges': to_number(df_raw['TotalCharges']),
        'partner': normalize_text(df_raw['Partner']),
        'dependents': normalize_text(df_raw['Dependents']),
        'phone_service': normalize_text(df_raw['PhoneService']),
        'multiple_lines': normalize_text(df_raw['MultipleLines']),
        'internet_service': normalize_text(df_raw['InternetService']),
        'online_security': normalize_text(df_raw['OnlineSecurity']),
        'online_backup': normalize_text(df_raw['OnlineBackup']),
        'device_protection': normalize_text(df_raw['DeviceProtection']),
        'tech_support': normalize_text(df_raw['TechSupport']),
        'streaming_tv': normalize_text(df_raw['StreamingTV']),
        'streaming_movies': normalize_text(df_raw['StreamingMovies']),
        'contract_type': normalize_text(df_raw['Contract']),
        'paperless_billing': normalize_text(df_raw['PaperlessBilling']),
        'payment_method': normalize_text(df_raw['PaymentMethod']),
    })

    df = (
        df[df['customer_id'].ne('')]
        .drop_duplicates(subset=['customer_id'])
        .reset_index(drop=True)
    )

    df['is_total_charges_missing'] = df['total_charges'].isna().astype('int64')
    df['has_partner'] = yes_no_to_int(df['partner'])
    df['has_dependents'] = yes_no_to_int(df['dependents'])
    df['paperless_billing_flag'] = yes_no_to_int(df['paperless_billing'])

    df['phone_service_flag'] = yes_no_to_int(df['phone_service'])
    df['multiple_lines_flag'] = np.where(
        df['phone_service_flag'].eq(0),
        0,
        yes_no_to_int(df['multiple_lines']),
    ).astype('int64')

    internet_service_l = df['internet_service'].str.lower()
    df['has_internet'] = np.where(internet_service_l.eq('no'), 0, 1).astype('int64')
    df['internet_is_dsl'] = internet_service_l.eq('dsl').astype('int64')
    df['internet_is_fiber'] = internet_service_l.eq('fiber optic').astype('int64')

    df['online_security_flag'] = yes_no_to_int(df['online_security'])
    df['online_backup_flag'] = yes_no_to_int(df['online_backup'])
    df['device_protection_flag'] = yes_no_to_int(df['device_protection'])
    df['tech_support_flag'] = yes_no_to_int(df['tech_support'])
    df['streaming_tv_flag'] = yes_no_to_int(df['streaming_tv'])
    df['streaming_movies_flag'] = yes_no_to_int(df['streaming_movies'])

    df['addon_services_cnt'] = (
        df['online_security_flag']
        + df['online_backup_flag']
        + df['device_protection_flag']
        + df['tech_support_flag']
        + df['streaming_tv_flag']
        + df['streaming_movies_flag']
    ).astype('int64')
    df['security_support_cnt'] = (
        df['online_security_flag'] + df['tech_support_flag']
    ).astype('int64')
    df['streaming_cnt'] = (
        df['streaming_tv_flag'] + df['streaming_movies_flag']
    ).astype('int64')

    df['contract_months'] = df['contract_type'].map({
        'Month-to-month': 1,
        'One year': 12,
        'Two year': 24,
    }).astype('float64')
    df['is_month_to_month'] = df['contract_type'].eq('Month-to-month').astype('int64')
    df['is_long_contract'] = df['contract_type'].isin(['One year', 'Two year']).astype('int64')
    df['is_auto_payment'] = df['payment_method'].str.contains('(automatic)', regex=False, na=False).astype('int64')
    df['is_electronic_check'] = df['payment_method'].eq('Electronic check').astype('int64')

    df['tenure_years'] = df['tenure_months'] / 12.0
    df['tenure_bucket'] = np.select(
        [
            df['tenure_months'].lt(6),
            df['tenure_months'].lt(12),
            df['tenure_months'].lt(24),
            df['tenure_months'].lt(48),
        ],
        ['0_5m', '6_11m', '12_23m', '24_47m'],
        default='48m_plus',
    )
    df['is_new_customer'] = df['tenure_months'].lt(6).astype('int64')

    df['total_charges_filled'] = df['total_charges'].fillna(
        df['monthly_charges'] * df['tenure_months']
    )
    denominator = np.where(df['tenure_months'].gt(0), df['tenure_months'], 1)
    df['avg_monthly_charge_lifetime'] = df['total_charges_filled'] / denominator
    df['charges_gap'] = df['total_charges_filled'] - (
        df['monthly_charges'] * df['tenure_months']
    )
    df['abs_charges_gap'] = df['charges_gap'].abs()
    df['monthly_charge_bucket'] = np.select(
        [df['monthly_charges'].lt(35), df['monthly_charges'].lt(70)],
        ['low', 'mid'],
        default='high',
    )

    feature_columns = [
        'customer_id',
        'gender',
        'internet_service',
        'contract_type',
        'payment_method',
        'tenure_bucket',
        'monthly_charge_bucket',
        'senior_citizen',
        'has_partner',
        'has_dependents',
        'tenure_months',
        'tenure_years',
        'is_new_customer',
        'phone_service_flag',
        'multiple_lines_flag',
        'has_internet',
        'internet_is_dsl',
        'internet_is_fiber',
        'online_security_flag',
        'online_backup_flag',
        'device_protection_flag',
        'tech_support_flag',
        'streaming_tv_flag',
        'streaming_movies_flag',
        'addon_services_cnt',
        'security_support_cnt',
        'streaming_cnt',
        'contract_months',
        'is_month_to_month',
        'is_long_contract',
        'paperless_billing_flag',
        'is_auto_payment',
        'is_electronic_check',
        'monthly_charges',
        'total_charges_filled',
        'avg_monthly_charge_lifetime',
        'is_total_charges_missing',
        'abs_charges_gap',
    ]

    return df[feature_columns]

In [ ]:
train_raw_df, inference_raw_df = train_test_split(
    raw_df,
    test_size=0.05,
    random_state=21,
    stratify=raw_df['Churn'],
)

features_df = engineer_customer_features(train_raw_df)
labels_df = make_labels(train_raw_df)
training_df = features_df.merge(labels_df, on='customer_id', how='inner')

print(f'Raw training rows: {train_raw_df.shape[0]:,}')
print(f'Inference holdout rows: {inference_raw_df.shape[0]:,}')
print(f'Modeling table rows: {training_df.shape[0]:,}')

display(training_df.head())
display(training_df['churn'].value_counts(normalize=True).rename('share').to_frame())

## 3. Train and Validation Split

This split happens inside the notebook. In the production pipeline, train and validation data become explicit Delta table contracts so validation can run as a separate workflow task.

In [ ]:
categorical_features = [
    'gender',
    'internet_service',
    'contract_type',
    'payment_method',
    'tenure_bucket',
    'monthly_charge_bucket',
]

target_column = 'churn'
id_column = 'customer_id'
feature_columns = [
    column for column in training_df.columns
    if column not in [id_column, target_column]
]
numeric_features = [
    'tenure_months',
    'tenure_years',
    'monthly_charges',
    'total_charges_filled',
    'avg_monthly_charge_lifetime',
    'abs_charges_gap',
]
passthrough_features = [
    column for column in feature_columns
    if column not in categorical_features + numeric_features
]

missing_model_features = [
    column for column in categorical_features + numeric_features
    if column not in feature_columns
]
if missing_model_features:
    raise ValueError(f'Missing model features: {missing_model_features}')

train_df, validation_df = train_test_split(
    training_df,
    test_size=0.20,
    random_state=123,
    stratify=training_df[target_column],
)

X_train = train_df[feature_columns]
y_train = train_df[target_column]
X_validation = validation_df[feature_columns]
y_validation = validation_df[target_column]

print(f'Training rows: {len(train_df):,}')
print(f'Validation rows: {len(validation_df):,}')
print(f'Numeric features: {len(numeric_features):,}')
print(f'Categorical features: {len(categorical_features):,}')
print(f'Passthrough features: {len(passthrough_features):,}')

## 4. Model Training

This section mirrors the scikit-learn training notebook in the productionized Databricks flow: `ColumnTransformer`, `StandardScaler`, `OrdinalEncoder`, `SelectKBest`, `LogisticRegression`, and `GridSearchCV`. It is convenient in a notebook, but the training choices, feature contract, tuning space, and validation policy are still hidden in notebook state.

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features),
], remainder='passthrough')

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest()),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, n_jobs=-1)),
])

param_grid = [
    {
        'classifier': [LogisticRegression(random_state=RANDOM_STATE, n_jobs=-1)],
        'classifier__max_iter': [500, 750, 1000],
        'feature_selection__k': ['all', 30, 15],
    }
]

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f'Best cross-validation accuracy: {grid_search.best_score_:.4f}')
print(f'Best parameters: {grid_search.best_params_}')

In [ ]:
cv_results = (
    pd.DataFrame(grid_search.cv_results_)
    .sort_values('rank_test_score')
    [[
        'rank_test_score',
        'mean_test_score',
        'std_test_score',
        'param_feature_selection__k',
        'param_classifier__max_iter',
    ]]
)

display(cv_results.head(10))

## 5. Model Validation

The current production project uses recall as the custom validation metric, with a minimum threshold of 0.5. This notebook mirrors that idea with a lightweight validation gate.

In [ ]:
best_model = grid_search.best_estimator_
validation_predictions = best_model.predict(X_validation)
validation_probabilities = best_model.predict_proba(X_validation)[:, 1]

validation_metrics = {
    'accuracy': accuracy_score(y_validation, validation_predictions),
    'precision': precision_score(y_validation, validation_predictions, zero_division=0),
    'recall': recall_score(y_validation, validation_predictions, zero_division=0),
    'roc_auc': roc_auc_score(y_validation, validation_probabilities),
}

metrics_df = pd.DataFrame.from_dict(
    validation_metrics,
    orient='index',
    columns=['value'],
)

confusion_df = pd.DataFrame(
    confusion_matrix(y_validation, validation_predictions),
    index=['actual_no_churn', 'actual_churn'],
    columns=['predicted_no_churn', 'predicted_churn'],
)

display(metrics_df.round(4))
display(confusion_df)
print('Classification report:')
print(classification_report(y_validation, validation_predictions, zero_division=0))

In [ ]:
validation_gate = pd.DataFrame([
    {
        'metric': metric,
        'value': validation_metrics[metric],
        'threshold': threshold,
        'passed': validation_metrics[metric] >= threshold,
    }
    for metric, threshold in VALIDATION_THRESHOLDS.items()
])

gate_status = 'PASSED' if validation_gate['passed'].all() else 'FAILED'
print(f'Model validation gate: {gate_status}')
display(validation_gate)

## Why This Is Not Production Ready Yet

This notebook is a strong discovery artifact, but weak as a production boundary:

- Data paths, schema assumptions, feature definitions, split strategy, tuning space, and validation thresholds are all embedded in cells.
- Feature logic is not packaged as tested code and cannot be reused safely by batch inference or monitoring.
- The validation data is created inside the same notebook that trains the model.
- There is no durable Feature Store or Unity Catalog table contract.
- There is no MLflow model registry promotion flow, deployment alias, CI/CD gate, scheduled job, or monitoring loop.

The rest of the repo productionizes this exact shape into Databricks Asset Bundles, Feature Engineering, MLflow, model validation, deployment, batch inference, and monitoring.